# EXH MMM — LOB Decomposition (Graft on Validated Grouped Model)
**Splits the validated grouped model's contributions to LOBs without re-estimating the media layer.**

A direct LOB-level refit was tried twice and its media layer **hurt out-of-sample validation**
(val R² fell *below* a controls-only baseline). The grouped model validates (Val R² 0.484) because
aggregation suppresses that overfitting. So instead of re-estimating below the group, this script
**decomposes** the grouped model's already-validated group contributions down to LOBs.

### Method
1. Compute each **group** (Funnel×LOB-group) contribution exactly as the grouped model does —
   the validated, untouched quantity.
2. Within each group, split that contribution to its LOBs using a **capped, model-free tilt**:
   * base = LOB's spend share within the group
   * tilt = LOB's **residual-attendance correlation** (spend vs attendance *after* removing the
     grouped model's control prediction — strips the film-slate/seasonality confound so we don't
     reward merely well-timed spend)
   * tilt capped at **±35%** of spend share; applied **only to curve-confidence LOBs**
   * borrow/frozen LOBs take **pure spend share** (their signal isn't trustworthy enough to tilt)
   * tilted shares renormalised → LOB contributions **sum exactly to the validated group total**

The refit contributes only its **confidence tiering** (which LOBs are curve / borrow / frozen), not
its parameters. The validated group totals are never overridden.

### Reads
* Grouped model artifacts: `response_params.pkl`, `spend_train_max_iter2.pkl`, `model_meta.pkl`,
  `model_spec.pkl`, `baseline_2025.pkl`, `df_full.parquet`
* Refit confidence tiers: `model_meta_refit.pkl` (LOB_TREAT)


In [0]:
import os, pickle
import numpy as np
import pandas as pd
from collections import defaultdict

GROUPED_PATH = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_Outcomes_v2/"
REFIT_PATH   = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_LOB_Refit/"
OUT_PATH     = "/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_EXH_LOB_Decomp_v2/"
os.makedirs(OUT_PATH, exist_ok=True)

# Grouped (validated) artifacts
with open(f"{GROUPED_PATH}response_params.pkl",       "rb") as f: response_params       = pickle.load(f)
with open(f"{GROUPED_PATH}spend_train_max_iter2.pkl", "rb") as f: spend_train_max_iter2 = pickle.load(f)
with open(f"{GROUPED_PATH}model_meta.pkl",            "rb") as f: model_meta            = pickle.load(f)
with open(f"{GROUPED_PATH}scaler_y.pkl",              "rb") as f: scaler_y              = pickle.load(f)
df_full = pd.read_parquet(f"{GROUPED_PATH}df_full.parquet")

ACTIVE_COMBOS = model_meta["ACTIVE_COMBOS"]
FUNNEL_COMBOS = model_meta["FUNNEL_COMBOS"]
LOB_GROUP_MAP = model_meta["LOB_GROUP_MAP"]
target_col    = model_meta["target_col"]
scale_factor  = float(scaler_y.scale_[0])
CONFIDENCE_TIERS = model_meta.get("CONFIDENCE_TIERS", {})

# Derive baseline_2025 from df_full (v2 pipeline doesn't export baseline_2025.pkl separately)
df_full["year"] = pd.to_datetime(df_full["week_date"]).dt.year
_funnels = ["Awareness", "Consideration", "Conversion"]
_media_cols = [c for c in df_full.columns if any(c.startswith(f"{f}_") for f in _funnels)
               and "__" not in c and "media_" not in c]

def _combo_cols(combo):
    """Return media columns belonging to a Funnel__Group combo."""
    funnel, group = combo.split("__", 1)
    lobs = LOB_GROUP_MAP.get(group, [])
    return [c for c in _media_cols if c.startswith(f"{funnel}_")
            and any(lob in c for lob in lobs)]

baseline_2025 = {}
df_2025_bl = df_full[df_full["year"] == 2025]
for combo in ACTIVE_COMBOS:
    cols = _combo_cols(combo)
    weekly_spend = df_2025_bl[cols].sum(axis=1)
    annual_spend = float(weekly_spend.sum())
    weeks_active = int((weekly_spend > 0).sum())
    weekly_avg   = annual_spend / 52
    baseline_2025[combo] = {
        "annual_spend": annual_spend,
        "weekly_avg": weekly_avg,
        "weeks_active": weeks_active,
    }

# Refit confidence tiers (curve / borrow / frozen) — the only thing we take from the refit
try:
    with open(f"{REFIT_PATH}model_meta_refit.pkl", "rb") as f: refit_meta = pickle.load(f)
    LOB_TREAT = refit_meta["LOB_TREAT"]
    print(f"Loaded refit confidence tiers for {len(LOB_TREAT)} LOBs")
except FileNotFoundError:
    LOB_TREAT = {}
    print("⚠ refit meta not found — all LOBs default to pure spend share (no tilt)")

print(f"Grouped model: {len(ACTIVE_COMBOS)} active combos | scale_factor={scale_factor:.1f}")

## 1. Validated Group Contributions (untouched)
Steady-state contribution per combo at 2025 spend, computed exactly as the grouped model does.

In [0]:
active_contributions = {}
for combo in ACTIVE_COMBOS:
    p   = response_params[combo]
    mx  = spend_train_max_iter2[combo]
    wk  = baseline_2025[combo]["weekly_avg"]
    s   = wk / mx
    ad  = s / (1 - p["alpha"])                       # steady-state adstock
    if p["saturation"] == "hill":
        sat = (ad ** p["alpha_s"]) / (ad ** p["alpha_s"] + p["k"] ** p["alpha_s"])
    else:
        sat = 1 - np.exp(-p["lam"] * ad)
    active_contributions[combo] = p["gamma"] * sat * scale_factor * 52

total_validated = sum(active_contributions.values())
print(f"Validated total media contribution (2025): {total_validated:,.0f}")
for combo in ACTIVE_COMBOS:
    print(f"  {combo:<40} {active_contributions[combo]:>12,.0f}")

# Funnel-average att-per-$ (from validated combos) — used to extrapolate genuinely-dropped LOBs
funnel_avg_rate = {}
for funnel, combos in FUNNEL_COMBOS.items():
    tc = sum(active_contributions[c] for c in combos if c in active_contributions)
    ts = sum(baseline_2025[c]["annual_spend"] for c in combos if c in baseline_2025)
    funnel_avg_rate[funnel] = tc / ts if ts > 0 else 0.0
print("\nFunnel-average att-per-$ (for extrapolating dropped LOBs):")
for f, r in funnel_avg_rate.items():
    print(f"  {f:<14} {r:.2f}")

# Same-GROUP att-per-$ (validated combos within each LOB-group, across funnels) for extrapolation.
# Preferred over funnel-average because it borrows from functional peers, not funnel-mates.
group_rate = {}
for combo in ACTIVE_COMBOS:
    grp = combo.split("__", 1)[1]   # combo = Funnel__Group
    group_rate.setdefault(grp, {"contrib": 0.0, "spend": 0.0})
    group_rate[grp]["contrib"] += active_contributions[combo]
    group_rate[grp]["spend"]   += baseline_2025[combo]["annual_spend"]
group_avg_rate = {g: (v["contrib"]/v["spend"] if v["spend"]>0 else 0.0)
                  for g, v in group_rate.items()}

# Funnel MEDIAN att-per-$ (robust fallback for orphan groups with no validated combo)
funnel_median_rate = {}
for funnel, combos in FUNNEL_COMBOS.items():
    rates = [active_contributions[c]/baseline_2025[c]["annual_spend"]
             for c in combos if c in active_contributions and baseline_2025[c]["annual_spend"]>0]
    funnel_median_rate[funnel] = float(np.median(rates)) if rates else 0.0

print("\nSame-group att-per-$ (preferred extrapolation rate):")
for g, r in group_avg_rate.items():
    print(f"  {g:<18} {r:.2f}")
print("Funnel-median att-per-$ (orphan fallback):")
for f, r in funnel_median_rate.items():
    print(f"  {f:<14} {r:.2f}")

## 1b. Reconciliation — are `__Other` LOBs a mapping bug or genuinely dropped?
For every LOB whose parent resolved to `__Other`, check whether the grouped model actually has a
validated combo for its funnel+group that the parser failed to match. This decides routing:
**mapping bug** → route to the real combo (full decomposition); **genuinely dropped** → extrapolate
via funnel-average att-per-$, flag as not-validated, and recommend a geo-test.

In [0]:
# Show the grouped model's actual structure
print("ACTIVE_COMBOS (validated):")
for c in ACTIVE_COMBOS: print(f"  {c}")
print("\nLOB_GROUP_MAP (group → member LOB labels):")
for g, lobs in LOB_GROUP_MAP.items():
    print(f"  {g:<18} {lobs}")

# For each (funnel, raw LOB) parsed from spend, does a validated combo exist?
def find_validated_combo(funnel, lob_raw):
    for group_name, lob_list in LOB_GROUP_MAP.items():
        if lob_raw in lob_list:
            combo = f"{funnel}__{group_name}"
            if combo in ACTIVE_COMBOS:
                return combo, group_name, "matched"
            else:
                return None, group_name, "group_found_combo_dropped"
    return None, None, "lob_not_in_any_group"

print("\n" + "="*78)
print("RECONCILIATION OF LOBs THAT PREVIOUSLY FELL TO __Other")
print("="*78)
# (we recompute parse below; here just diagnose using the same parser)
_media = [c for c in df_full.columns if any(c.startswith(f"{f}_") for f in
          ["Awareness","Consideration","Conversion"]) and "__" not in c and "media_" not in c]
_fl = {}
for col in _media:
    parts = col.split("_",1); _fl.setdefault((parts[0], parts[1][:parts[1].rfind("_")]), []).append(col)

reasons = {}
for (funnel, lob_raw), cols in _fl.items():
    spend = float(df_full[df_full["year"]==2025][cols].sum().sum())
    if spend <= 0: continue
    combo, group, reason = find_validated_combo(funnel, lob_raw)
    if combo is None:
        reasons[(funnel, lob_raw)] = reason
        print(f"  {funnel:<14} {lob_raw:<34} ${spend:>10,.0f}  → {reason}"
              + (f"  (group '{group}' exists but {funnel}__{group} not in ACTIVE_COMBOS)"
                 if reason=="group_found_combo_dropped" else ""))
if not reasons:
    print("  none — every spending LOB maps to a validated combo. No __Other cases.")

## 2. Residual Attendance (control confound removed)
Tilt signal must not reward slate-timed spend. We build residual attendance = actual − the grouped
model's **control-only** prediction, so the correlation reflects only the part of attendance the
controls don't already explain.

In [0]:
# Reconstruct control-only prediction from the grouped model's saved fields if present,
# else fall back to df_full's stored controls prediction.
if "controls_pred" in df_full.columns:
    control_pred = df_full["controls_pred"].values
    print("Using stored controls_pred from grouped model.")
elif "pred" in df_full.columns and "total_media" in df_full.columns:
    control_pred = (df_full["pred"] - df_full["total_media"]).values
    print("Derived controls_pred = pred − total_media.")
else:
    raise KeyError("Cannot find control prediction. Need 'controls_pred' or "
                   "('pred' and 'total_media') in df_full.")

residual_att = df_full[target_col].values - control_pred
print(f"Residual attendance: mean {residual_att.mean():,.0f} | std {residual_att.std():,.0f}")

## 3. Parse LOB Spend & Compute Per-LOB Tilt Signal
Same parser as the grouped contribution script. For each (funnel, LOB) we compute the correlation
between its weekly spend and residual attendance — the model-free, confound-stripped tilt signal.

In [0]:
media_cols = [c for c in df_full.columns if any(
    c.startswith(f"{f}_") for f in ["Awareness","Consideration","Conversion"]
) and "__" not in c and "media_" not in c]

funnel_lob_cols = defaultdict(list)
for col in media_cols:
    parts = col.split("_", 1); funnel = parts[0]; rest = parts[1]
    funnel_lob_cols[(funnel, rest[:rest.rfind("_")])].append(col)

# Weekly spend per (funnel, lob); full training-period spend; residual correlation
# NOTE: Use full training period spend (not just 2025) so LOBs with historical
# spend but no current-year spend still get their share of the learned response.
lob_spend_2025, lob_tilt_corr = {}, {}
for (funnel, lob), cols in funnel_lob_cols.items():
    weekly = df_full[cols].sum(axis=1).values.astype(float)
    lob_spend_2025[(funnel, lob)] = float(df_full[cols].sum().sum())
    # correlation only over weeks the LOB was active (else zeros dominate and bias toward 0)
    active = weekly > 0
    if active.sum() >= 5 and np.std(weekly[active]) > 0 and np.std(residual_att[active]) > 0:
        lob_tilt_corr[(funnel, lob)] = float(np.corrcoef(weekly[active], residual_att[active])[0, 1])
    else:
        lob_tilt_corr[(funnel, lob)] = 0.0   # insufficient signal → no tilt

print(f"Parsed {len(funnel_lob_cols)} (funnel, LOB) cells")
print("\nResidual-attendance correlation by cell (tilt signal):")
for k in sorted(lob_tilt_corr, key=lambda x: -abs(lob_tilt_corr[x]))[:15]:
    print(f"  {k[0]:<14} {k[1]:<34} r={lob_tilt_corr[k]:+.2f}  ${lob_spend_2025[k]:>10,.0f}")

## 4. Tilted Decomposition to LOBs
For each combo, split its validated contribution across member LOBs: base spend share, tilted by
residual correlation (curve LOBs only), capped at ±35%, renormalised to preserve the group total.

In [0]:
TILT_CAP = 0.35   # max fractional move from spend share

def lob_display_name(raw):
    return {"EXH__Brand":"EXH - Brand","FandB":"F&B","GandE":"G&E",
            "Film_Marketing__Inform":"Film Marketing - Inform",
            "Performance__Convert_and_Upgrade":"Performance - Convert & Upgrade",
            "Premium_Experiences":"Premium Experiences - AVX",
            "Premium_Experiences__AVX":"Premium Experiences - AVX",
            "Premium_Experiences__IMAX":"Premium Experiences - IMAX",
            "Premium_Experiences__VIP":"Premium Experiences - VIP",
            "Prepaid__B2B":"Prepaid - B2B","Prepaid__B2C":"Prepaid - B2C",
            "Cineclub":"Cineclub"}.get(raw, raw)

def get_parent_combo(funnel, lob_raw):
    """Returns (combo_or_None, group, routing) — routing in {validated, dropped, no_group}."""
    for group_name, lob_list in LOB_GROUP_MAP.items():
        if lob_raw in lob_list:
            combo = f"{funnel}__{group_name}"
            if combo in ACTIVE_COMBOS:
                return combo, group_name, "validated"
            return None, group_name, "dropped"
    return None, None, "no_group"

# Group spending cells by parent combo; track dropped/no-group separately
combo_members = defaultdict(list)
extrapolate_cells = []   # (funnel, lob_raw, spend) for genuinely-dropped LOBs
for (funnel, lob_raw), spend in lob_spend_2025.items():
    if spend <= 0:
        continue
    combo, group, routing = get_parent_combo(funnel, lob_raw)
    if routing == "validated":
        combo_members[combo].append((funnel, lob_raw, spend))
    else:
        extrapolate_cells.append((funnel, lob_raw, spend))  # dropped or ungrouped

# Geo-test list: crucial-but-unvalidated LOBs
GEO_TEST_LOBS = set()

rows = []
# ── (A) Validated combos → tilted decomposition (sums to validated total) ──────
for combo, members in combo_members.items():
    group_total_spend  = sum(s for _, _, s in members)
    group_contribution = active_contributions[combo]
    base = {(f, l): (s / group_total_spend if group_total_spend > 0 else 1/len(members))
            for f, l, s in members}
    tilt_factor = {}
    for f, l, s in members:
        disp = lob_display_name(l)
        treat = LOB_TREAT.get(disp, "curve")
        if treat == "curve":
            r = lob_tilt_corr.get((f, l), 0.0)
            tilt_factor[(f, l)] = 1.0 + TILT_CAP * max(-1.0, min(1.0, r))
        else:
            tilt_factor[(f, l)] = 1.0
    raw_w = {k: base[k] * tilt_factor[k] for k in base}
    wsum  = sum(raw_w.values())
    shares = {k: raw_w[k] / wsum for k in raw_w}
    for f, l, s in members:
        disp = lob_display_name(l)
        treat = LOB_TREAT.get(disp, "curve")
        share = shares[(f, l)]
        contrib = group_contribution * share
        rows.append({
            "Funnel": f, "LOB": disp, "Parent Combo": combo, "Status": "Decomposed (validated)",
            "Treatment": treat,
            "Reallocatable": "YES" if treat == "curve" else
                             ("directional" if treat == "borrow" else "frozen"),
            "Annual Spend": round(s, 0),
            "Spend Share (grp)": round(base[(f, l)] * 100, 1),
            "Tilt r": round(lob_tilt_corr.get((f, l), 0.0), 2) if treat == "curve" else None,
            "Final Share (grp)": round(share * 100, 1),
            "Annual Contrib": round(contrib, 0),
            "Att per $1": round(contrib / s, 2) if s > 0 else None,
        })

# ── (B) Genuinely-dropped LOBs → extrapolate via SAME-GROUP att-per-$ ─────────
# Borrow from functional peers (the LOB's own group) rather than the funnel blend, which
# imported Premium_Brand's high efficiency onto unrelated LOBs. Orphan groups (no validated
# combo anywhere) fall back to the funnel MEDIAN (robust to outliers).
# Track per-LOB extrapolated spend by funnel to decide the dominant-funnel geo-test flag.
from collections import defaultdict as _dd
extrap_by_lob = _dd(lambda: {"spend": 0.0, "contrib": 0.0, "funnels": set(), "rate_src": ""})
lob_total_spend_all = _dd(float)  # all spend (validated + extrap) per LOB, for dominance test
for (ff, ll), sp in lob_spend_2025.items():
    if sp > 0: lob_total_spend_all[lob_display_name(ll)] += sp

for f, l, s in extrapolate_cells:
    disp = lob_display_name(l)
    _, grp, _ = get_parent_combo(f, l)   # group name even if combo dropped
    if grp is not None and grp in group_avg_rate and group_avg_rate[grp] > 0:
        rate = group_avg_rate[grp]; src = f"group:{grp}"
    else:
        rate = funnel_median_rate.get(f, 0.0); src = f"median:{f}"
    contrib = s * rate
    extrap_by_lob[disp]["spend"]   += s
    extrap_by_lob[disp]["contrib"] += contrib
    extrap_by_lob[disp]["funnels"].add(f)
    extrap_by_lob[disp]["rate_src"] = src
    rows.append({
        "Funnel": f, "LOB": disp, "Parent Combo": f"{f}__Other",
        "Status": "Extrapolated ⚠ (not model-validated)",
        "Treatment": "extrapolated",
        "Reallocatable": "NO — geo-test",
        "Annual Spend": round(s, 0),
        "Spend Share (grp)": None,
        "Tilt r": None,
        "Final Share (grp)": None,
        "Annual Contrib": round(contrib, 0),
        "Att per $1": round(rate, 2),
    })

# Two-tier geo-test:
#  HEADLINE  = LOBs with ≥35% of spend unvalidated → genuinely under-measured, test priority
#  SLIVERS   = LOBs that are otherwise validated but have a small extrapolated sliver (disclosed only)
GEO_TEST_HEADLINE, GEO_TEST_SLIVERS = set(), set()
for disp, info in extrap_by_lob.items():
    total_lob = lob_total_spend_all[disp]
    extrap_share = info["spend"] / total_lob if total_lob > 0 else 1.0
    if extrap_share >= 0.35:           # ≥35% of the LOB's spend is unvalidated → test
        GEO_TEST_HEADLINE.add(disp)
    else:
        GEO_TEST_SLIVERS.add(disp)
GEO_TEST_LOBS = GEO_TEST_HEADLINE     # keep name for downstream export

decomp = pd.DataFrame(rows)

# Two totals: validated (sums exactly) and validated+extrapolated (planning figure)
validated_contrib  = decomp.loc[decomp["Status"].str.startswith("Decomposed"), "Annual Contrib"].sum()
extrapolated_contrib = decomp.loc[decomp["Status"].str.startswith("Extrapolated"), "Annual Contrib"].sum()
print(f"Validated decomposed total : {validated_contrib:,.0f}  (should match {total_validated:,.0f})")
print(f"Extrapolated (dropped) add : {extrapolated_contrib:,.0f}")
print(f"Planning total (val+extrap): {validated_contrib + extrapolated_contrib:,.0f}")
print(f"\nValidated sum-back diff: {validated_contrib - total_validated:,.0f}  (≈0 expected)")
if GEO_TEST_HEADLINE:
    print(f"\n⚠ GEO-TEST (headline — dominant funnel unvalidated): {sorted(GEO_TEST_HEADLINE)}")
if GEO_TEST_SLIVERS:
    print(f"  (disclosed slivers only — main curve is validated, NOT a test priority): {sorted(GEO_TEST_SLIVERS)}")

## 5. Per-LOB Output Table
Sorted by contribution, flagged by reallocation eligibility. Curve LOBs (residual-correlation
tilted) are reallocatable; borrow LOBs are directional; frozen are report-only. Every contribution
is a slice of a **validated** group total.

In [0]:
show = decomp.copy()
total_spend = show["Annual Spend"].sum()
total_contrib = show["Annual Contrib"].dropna().sum()
show["Spend %"]   = (show["Annual Spend"] / total_spend * 100).round(1)
show["Contrib %"] = (show["Annual Contrib"] / total_contrib * 100).round(1)

disp = show.sort_values("Annual Contrib", ascending=False, na_position="last").copy()
disp["Annual Spend"]   = disp["Annual Spend"].apply(lambda x: f"${x:,.0f}")
disp["Annual Contrib"] = disp["Annual Contrib"].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "—")
disp["Att per $1"]     = disp["Att per $1"].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "—")
display(disp[["Funnel","LOB","Parent Combo","Status","Reallocatable","Annual Spend","Spend %",
              "Tilt r","Final Share (grp)","Annual Contrib","Contrib %","Att per $1"]])

print(f"\nReallocatable (curve) LOBs carry residual-correlation tilts (capped ±{int(TILT_CAP*100)}%).")
print("Directional (borrow) and frozen LOBs use pure spend share — no tilt.")

In [0]:
# ── Flighting prerequisites ───────────────────────────────────────────────────
FUNNELS = ["Awareness", "Consideration", "Conversion"]

CPX_GREEN = "#2E7D32"
CPX_MID   = "#AABBCC"
CPX_DGREY = "#444444"

funnel_decay = {}
for f in FUNNELS:
    alphas = [p["alpha"] for combo, p in response_params.items()
              if combo.startswith(f) and "alpha" in p]
    funnel_decay[f] = float(np.mean(alphas)) if alphas else 0.5

def optimal_lag_range(decay):
    hl = max(1.0, np.log(0.5) / np.log(max(decay, 0.01)))
    return max(1, int(np.floor(hl * 0.5))), min(8, int(np.ceil(hl * 1.5)))

funnel_lags    = {f: optimal_lag_range(funnel_decay[f]) for f in FUNNELS}
PEAK_WKS       = [1, 11, 33, 49, 50]
TROUGH_WKS     = list(range(19, 24)) + list(range(36, 42))

def get_optimal_weeks(lag_min, lag_max):
    wks_set = set()
    for p in PEAK_WKS:
        for lag in range(lag_min, lag_max + 1):
            w = p - lag
            if w <= 0: w += 52
            wks_set.add(w)
    return sorted(wks_set)

funnel_opt_wks = {f: get_optimal_weeks(*funnel_lags[f]) for f in FUNNELS}
df_fl          = df_full[df_full["year"] != 2026].copy()

for f in FUNNELS:
    print(f"  {f}: decay={funnel_decay[f]:.3f}  lag={funnel_lags[f]}  opt_wks={funnel_opt_wks[f]}")

## 6. How Much the Tilt Actually Moved Things
Transparency check: compares each curve LOB's pure-spend-share contribution vs its tilted
contribution, so you can see exactly how much the residual-correlation signal shifted allocations
inside each group. Large moves on thin LOBs are the ones to scrutinise.

In [0]:
moves = []
for combo, members in combo_members.items():
    if combo not in active_contributions:
        continue
    gt   = active_contributions[combo]
    gts  = sum(s for _, _, s in members)
    for f, l, s in members:
        disp_l = lob_display_name(l)
        pure   = gt * (s / gts if gts > 0 else 1/len(members))
        row    = decomp[(decomp["Funnel"] == f) & (decomp["LOB"] == disp_l) &
                        (decomp["Parent Combo"] == combo)]
        if not len(row): continue
        tilted = row["Annual Contrib"].iloc[0]
        if pd.isna(tilted): continue
        moves.append({"Funnel": f, "LOB": disp_l, "Parent": combo,
                      "Pure-share Contrib": round(pure, 0),
                      "Tilted Contrib": round(tilted, 0),
                      "Δ": round(tilted - pure, 0),
                      "Δ %": round((tilted - pure) / pure * 100, 1) if pure > 0 else 0})
moves_df = pd.DataFrame(moves).sort_values("Δ %", key=lambda s: s.abs(), ascending=False)
display(moves_df)
print(f"\nMax absolute move: {moves_df['Δ %'].abs().max():.1f}% "
      f"(capped at ±{int(TILT_CAP*100)}% before renormalisation; renorm can push slightly beyond).")

In [0]:
# ── Export ────────────────────────────────────────────────────────────────────
decomp.to_csv(f"{OUT_PATH}lob_decomposition.csv", index=False)
moves_df.to_csv(f"{OUT_PATH}tilt_movement_audit.csv", index=False)

decomp_artifact = {
    "decomp_table": decomp,
    "validated_group_contributions": active_contributions,
    "tilt_cap": TILT_CAP,
    "tilt_signal": "residual_attendance_correlation",
    "lob_treat": LOB_TREAT,
    "reallocatable_lobs": sorted(decomp.loc[decomp["Reallocatable"] == "YES", "LOB"].unique()),
    "geo_test_headline": sorted(GEO_TEST_HEADLINE),
    "geo_test_slivers": sorted(GEO_TEST_SLIVERS),
    "validated_total": float(validated_contrib),
    "extrapolated_total": float(extrapolated_contrib),
}
with open(f"{OUT_PATH}lob_decomposition.pkl", "wb") as f:
    pickle.dump(decomp_artifact, f)

print(f"Saved to {OUT_PATH}:")
print("  • lob_decomposition.csv   (per-LOB contributions, slices of validated totals)")
print("  • tilt_movement_audit.csv (pure-share vs tilted, transparency)")
print("  • lob_decomposition.pkl   (for the optimizer script)")
print(f"\nReallocatable LOBs: {decomp_artifact['reallocatable_lobs']}")
print(f"Geo-test (headline, ≥35% unvalidated): {decomp_artifact['geo_test_headline']}")
print(f"Disclosed slivers (<35%, main curve validated): {decomp_artifact['geo_test_slivers']}")
print("\nThese contributions sum to the validated group total and never override it.")
print("The optimizer (next script) reallocates only across the curve/reallocatable LOBs.")

In [0]:
# ── CELL: Per-LOB Annual Spend & Attendance Contribution — grouped bars by year ──
# One chart per LOB. For each year: stacked spend bar + attendance contribution bar side by side.
# Unvalidated/geo-test LOBs included with funnel-avg att-per-$1 as directional proxy.
# Funnel colours: Awareness=#002B5C, Consideration=#E31837, Conversion=#F5A800

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import os

CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_DGREY = "#4A4A4A"
CPX_LGREY = "#CCCCCC"

FUNNEL_COLOR = {"Awareness": CPX_NAVY, "Consideration": CPX_RED, "Conversion": CPX_GOLD}
FUNNELS      = ["Awareness", "Consideration", "Conversion"]

def save_fig(fig, fname):
    fig.savefig(os.path.join(OUT_PATH, fname), dpi=150, bbox_inches="tight")
    print(f"Saved: {fname}")

# ── 1. Validated att-per-$1 from decomp ──────────────────────────────────────
decomp_valid = decomp[decomp["Status"].str.startswith("Decomposed")].copy()

fl_rate = {}
for (funnel, lob), grp in decomp_valid.groupby(["Funnel", "LOB"]):
    valid = grp[grp["Att per $1"].notna() & (grp["Annual Spend"] > 0)]
    if valid.empty:
        fl_rate[(funnel, lob)] = 0.0
    else:
        fl_rate[(funnel, lob)] = (
            (valid["Att per $1"] * valid["Annual Spend"]).sum() / valid["Annual Spend"].sum()
        )

# ── 2. Fallback: funnel-average rate for unvalidated/geo-test LOBs ───────────
funnel_avg_rate = {}
for funnel in FUNNELS:
    rates = [v for (f, _), v in fl_rate.items() if f == funnel and v > 0]
    funnel_avg_rate[funnel] = float(np.mean(rates)) if rates else 0.0

years_avail = sorted(y for y in df_full["year"].dropna().unique().astype(int) if y < 2026)

directional_lobs = set()
for (funnel, lob_raw), cols in funnel_lob_cols.items():
    disp = lob_display_name(lob_raw)
    key  = (funnel, disp)
    if key not in fl_rate:
        has_spend = any(
            float(df_full[df_full["year"] == yr][cols].sum().sum()) > 0
            for yr in years_avail
        )
        if has_spend:
            fl_rate[key] = funnel_avg_rate.get(funnel, 0.0)
            directional_lobs.add(key)

# ── 3. Annual spend per (funnel, LOB, year) — all LOBs ───────────────────────
fl_yr_spend = {}
for (funnel, lob_raw), cols in funnel_lob_cols.items():
    disp = lob_display_name(lob_raw)
    key  = (funnel, disp)
    if key not in fl_rate:
        continue
    if key not in fl_yr_spend:
        fl_yr_spend[key] = {yr: 0.0 for yr in years_avail}
    for yr in years_avail:
        fl_yr_spend[key][yr] += float(df_full[df_full["year"] == yr][cols].sum().sum())

# ── 4. Unique LOBs ────────────────────────────────────────────────────────────
all_lobs = sorted({lob for (_, lob) in fl_rate})

# ── 5. One chart per LOB ─────────────────────────────────────────────────────
n_yrs = len(years_avail)
bar_w = 0.35
gap   = 0.15
group = bar_w * 2 + gap + 0.2

for lob in all_lobs:
    active_funnels = [f for f in FUNNELS if (f, lob) in fl_rate and
                      any(fl_yr_spend.get((f, lob), {}).get(yr, 0) > 0 for yr in years_avail)]
    if not active_funnels:
        continue

    is_directional = any((f, lob) in directional_lobs for f in active_funnels)

    spend_by_f   = {f: np.array([fl_yr_spend.get((f, lob), {}).get(yr, 0) / 1e6
                                  for yr in years_avail]) for f in active_funnels}
    contrib_by_f = {f: np.array([fl_yr_spend.get((f, lob), {}).get(yr, 0) * fl_rate[(f, lob)] / 1e6
                                  for yr in years_avail]) for f in active_funnels}

    total_spend_yr   = sum(spend_by_f[f]  for f in active_funnels)
    total_contrib_yr = sum(contrib_by_f[f] for f in active_funnels)

    x_spend   = np.array([i * group for i in range(n_yrs)])
    x_contrib = x_spend + bar_w + gap

    fig, ax_s = plt.subplots(figsize=(max(7, n_yrs * 2.5), 5))
    ax_c = ax_s.twinx()

    # Stacked spend bars (left axis)
    bottoms_s = np.zeros(n_yrs)
    for f in active_funnels:
        ax_s.bar(x_spend, spend_by_f[f], bar_w, bottom=bottoms_s,
                 color=FUNNEL_COLOR[f], alpha=0.9, edgecolor="white", lw=0.4)
        bottoms_s += spend_by_f[f]

    # Stacked contribution bars (right axis) — hatched
    bottoms_c = np.zeros(n_yrs)
    for f in active_funnels:
        ax_c.bar(x_contrib, contrib_by_f[f], bar_w, bottom=bottoms_c,
                 color=FUNNEL_COLOR[f],
                 alpha=0.35 if is_directional else 0.55,
                 edgecolor="white", lw=0.4, hatch="///")
        bottoms_c += contrib_by_f[f]

    # Value labels
    for i in range(n_yrs):
        if bottoms_s[i] > 0:
            ax_s.annotate(f"${bottoms_s[i]:.1f}M",
                          xy=(x_spend[i], bottoms_s[i]),
                          xytext=(0, 4), textcoords="offset points",
                          ha="center", fontsize=7.5, color=CPX_DGREY, fontweight="bold")
        if bottoms_c[i] > 0:
            ax_c.annotate(f"{bottoms_c[i]:.2f}M" + (" ~" if is_directional else ""),
                          xy=(x_contrib[i], bottoms_c[i]),
                          xytext=(0, 4), textcoords="offset points",
                          ha="center", fontsize=7.5,
                          color=CPX_RED if is_directional else CPX_DGREY,
                          fontweight="bold")

    # X-axis
    x_mid = (x_spend + x_contrib) / 2
    ax_s.set_xticks(x_mid)
    ax_s.set_xticklabels([str(y) for y in years_avail], fontsize=10, color=CPX_DGREY)
    ax_s.set_xlim(-0.3, x_contrib[-1] + bar_w + 0.3)

    # Axis formatting
    ax_s.set_ylabel("Annual Media Spend ($M)", fontsize=9, color=CPX_DGREY)
    ax_s.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:.1f}M"))
    ax_s.set_ylim(bottom=0, top=max(bottoms_s.max(), 0.01) * 1.3)

    ax_c.set_ylabel("Media Attendance Contribution (M)", fontsize=9, color=CPX_DGREY)
    ax_c.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.2f}M"))
    ax_c.set_ylim(bottom=0, top=max(bottoms_c.max(), 0.01) * 1.3)

    for ax in (ax_s, ax_c):
        ax.spines["top"].set_visible(False)
        ax.tick_params(axis="y", labelsize=8)

    ax_s.set_axisbelow(True)
    ax_s.grid(axis="y", alpha=0.2, color=CPX_LGREY)

    # Legend
    handles = [mpatches.Patch(color=FUNNEL_COLOR[f], label=f) for f in active_funnels]
    handles += [
        mpatches.Patch(facecolor="#999999", alpha=0.9, label="Spend"),
        mpatches.Patch(facecolor="#999999", alpha=0.55, hatch="///", label="Attendance Contrib"),
    ]
    ax_s.legend(handles=handles, fontsize=8,
                loc="upper left", bbox_to_anchor=(1.18, 1),
                borderaxespad=0, framealpha=0.9, edgecolor=CPX_LGREY)

    ax_s.set_title(
        f"{lob}\nAnnual Spend vs Attendance Contribution",
        fontsize=11, fontweight="bold",
        color=CPX_NAVY,
        pad=10
    )

    # Att-per-$1 annotation
    rate_lines = "\n".join(
        f"{f}: {fl_rate[(f, lob)]:.2f} att/$1"
        + (" (directional ⚠)" if (f, lob) in directional_lobs else "")
        for f in active_funnels
    )
    ax_s.text(
        0.99, 0.97, rate_lines,
        transform=ax_s.transAxes,
        ha="right", va="top",
        fontsize=8, color=CPX_DGREY, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor=CPX_LGREY,
                  alpha=0.9),
    )

    plt.tight_layout()
    plt.subplots_adjust(right=0.68)
    safe_name = lob.replace(" ", "_").replace("/", "_").replace("&", "and")
    save_fig(fig, f"I2_B9_LOB_{safe_name}_SpendContrib.png")
    plt.show()
    plt.close(fig)

In [0]:
# ── 4b. Per-year attendance structure ────────────────────────────────────────
year_att_structure = {}
for yr in years_avail:
    df_yr      = df_full[df_full["year"] == yr]
    actual_yr  = float(df_yr[model_meta["target_col"]].sum())
    pred_yr    = float(df_yr["pred"].sum())
    recon_yr   = actual_yr / pred_yr if pred_yr > 0 else 1.0

    # controls_pred and total_media are the model's own decomposition
    # Reconcile both to actual using the same recon factor
    controls_yr = float(df_yr["controls_pred"].sum()) * recon_yr
    media_yr    = float(df_yr["total_media"].sum())   * recon_yr

    # Directional share = extrapolated LOB contributions from graft (within validated media)
    # Scale 2025 graft extrapolated contrib by year's media ratio
    graft_dir_2025 = float(
        decomp[decomp["Status"].str.startswith("Extrapolated")]["Annual Contrib"]
        .dropna().sum()
    )
    graft_val_2025 = float(
        decomp[decomp["Status"].str.startswith("Decomposed")]["Annual Contrib"]
        .dropna().sum()
    )
    graft_total_2025 = graft_dir_2025 + graft_val_2025
    dir_share = graft_dir_2025 / graft_total_2025 if graft_total_2025 > 0 else 0.0

    year_att_structure[yr] = {
        "actual"           : actual_yr,
        "media"            : media_yr,
        "controls"         : controls_yr,
        "validated"        : media_yr * (1 - dir_share),
        "directional_graft": media_yr * dir_share,
    }

    validated_yr   = year_att_structure[yr]["validated"]
    directional_yr = year_att_structure[yr]["directional_graft"]
    print(f"{yr}  actual={actual_yr/1e6:.2f}M  "
          f"controls={controls_yr/1e6:.2f}M ({controls_yr/actual_yr*100:.1f}%)  "
          f"media={media_yr/1e6:.2f}M ({media_yr/actual_yr*100:.1f}%)  "
          f"validated={validated_yr/1e6:.2f}M ({validated_yr/actual_yr*100:.1f}%)  "
          f"directional={max(directional_yr,0)/1e6:.2f}M")

In [0]:
# ── CELL: Annual Media Contribution by Original LOB — grouped by year, sourced from decomp ──
# Replicates the grouped-model chart using decomp-derived contributions.
# One bar per (funnel, LOB) per year, coloured by funnel.
# Unvalidated/geo-test LOBs included with funnel-avg att-per-$1 as directional proxy (hatched).

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import os

CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_DGREY = "#4A4A4A"
CPX_LGREY = "#CCCCCC"

FUNNEL_COLORS = {"Awareness": CPX_NAVY, "Consideration": CPX_RED, "Conversion": CPX_GOLD}
FUNNELS       = ["Awareness", "Consideration", "Conversion"]

def save_fig(fig, fname):
    fig.savefig(os.path.join(OUT_PATH, fname), dpi=150, bbox_inches="tight")
    print(f"Saved: {fname}")

# ── 1. Validated att-per-$1 from decomp ──────────────────────────────────────
decomp_valid = decomp[decomp["Status"].str.startswith("Decomposed")].copy()

fl_rate = {}
for (funnel, lob), grp in decomp_valid.groupby(["Funnel", "LOB"]):
    valid = grp[grp["Att per $1"].notna() & (grp["Annual Spend"] > 0)]
    if valid.empty:
        fl_rate[(funnel, lob)] = 0.0
    else:
        fl_rate[(funnel, lob)] = (
            (valid["Att per $1"] * valid["Annual Spend"]).sum() / valid["Annual Spend"].sum()
        )

# ── 2. Fallback: funnel-average rate for unvalidated/geo-test LOBs ───────────
funnel_avg_rate = {}
for funnel in FUNNELS:
    rates = [v for (f, _), v in fl_rate.items() if f == funnel and v > 0]
    funnel_avg_rate[funnel] = float(np.mean(rates)) if rates else 0.0

years_avail = sorted(y for y in df_full["year"].dropna().unique().astype(int) if y < 2026)

directional_lobs = set()
for (funnel, lob_raw), cols in funnel_lob_cols.items():
    disp = lob_display_name(lob_raw)
    key  = (funnel, disp)
    if key not in fl_rate:
        has_spend = any(
            float(df_full[df_full["year"] == yr][cols].sum().sum()) > 0
            for yr in years_avail
        )
        if has_spend:
            fl_rate[key] = funnel_avg_rate.get(funnel, 0.0)
            directional_lobs.add(key)

# ── 3. Annual spend per (funnel, LOB, year) — all LOBs ───────────────────────
fl_yr_spend = {}
for (funnel, lob_raw), cols in funnel_lob_cols.items():
    disp = lob_display_name(lob_raw)
    key  = (funnel, disp)
    if key not in fl_rate:
        continue
    if key not in fl_yr_spend:
        fl_yr_spend[key] = {yr: 0.0 for yr in years_avail}
    for yr in years_avail:
        fl_yr_spend[key][yr] += float(df_full[df_full["year"] == yr][cols].sum().sum())

# ── 4. Ordered LOB list — validated first, then directional ──────────────────
ORIG_LOBS = [
    {"funnel": f, "lob": lob,
     "color": FUNNEL_COLORS[f],
     "directional": (f, lob) in directional_lobs}
    for f in FUNNELS
    for lob in sorted({l for (fn, l) in fl_rate if fn == f})
]

# ── 5. One chart per year ─────────────────────────────────────────────────────
n_lobs  = len(ORIG_LOBS)
bar_w   = 0.7
x_pos   = np.arange(n_lobs)

for yr in years_avail:
    fig, ax = plt.subplots(figsize=(18, 9))
    legend_handles = {f: None for f in FUNNELS}
    legend_handle_dir = None

    for l_idx, lob_item in enumerate(ORIG_LOBS):
        f, lob = lob_item["funnel"], lob_item["lob"]
        is_dir = lob_item["directional"]
        val    = fl_yr_spend.get((f, lob), {}).get(yr, 0.0) * fl_rate.get((f, lob), 0.0)

        bars = ax.bar(x_pos[l_idx], val / 1e6, width=bar_w * 0.85,
                      color=lob_item["color"],
                      alpha=0.4 if is_dir else 0.85,
                      hatch="///" if is_dir else None,
                      edgecolor="white", linewidth=0.4)

        if legend_handles[f] is None:
            legend_handles[f] = bars
        if is_dir and legend_handle_dir is None:
            legend_handle_dir = bars

        # Value label
        if val / 1e6 > 0.02:
            ax.text(x_pos[l_idx], val / 1e6 + 0.01,
                    f"{val/1e6:.2f}M~" if is_dir else f"{val/1e6:.2f}M",
                    ha="center", va="bottom", fontsize=11,
                    color=CPX_RED if is_dir else CPX_DGREY,
                    fontweight="bold")

    # X-axis LOB labels
    lob_labels = [
        li["lob"] + (" ⚠" if li["directional"] else "")
        for li in ORIG_LOBS
    ]
    ax.set_xticks(x_pos)
    ax.set_xticklabels(lob_labels, rotation=45, ha="right", fontsize=11,
                       color=CPX_DGREY)

    # Axes & formatting
    ax.set_ylabel("Media Attendance Contribution (M)", fontsize=13, color=CPX_DGREY)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.2f}M"))
    ax.set_ylim(0, ax.get_ylim()[1] * 1.25)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.2, color=CPX_LGREY)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="y", labelsize=11)

    ax.set_title(
        f"Annual Media Contribution by Original LOB — {yr}\n"
        "Media effect only  |  ⚠ Hatched = directional, funnel-avg rate applied",
        fontsize=15, fontweight="bold", color=CPX_NAVY,
    )

    # Legend
    leg_handles = [
        mpatches.Patch(facecolor=FUNNEL_COLORS[f], alpha=0.85, label=f)
        for f in FUNNELS
        if any(li["funnel"] == f for li in ORIG_LOBS
               if fl_yr_spend.get((li["funnel"], li["lob"]), {}).get(yr, 0.0) > 0)
    ]
    leg_labels = [p.get_label() for p in leg_handles]

    if any(li["directional"] for li in ORIG_LOBS):
        leg_handles.append(mpatches.Patch(facecolor=CPX_DGREY, alpha=0.4, hatch="///",
                                          label="Directional ⚠ (unvalidated)"))
        leg_labels.append("Directional ⚠ (unvalidated)")

    ax.legend(handles=leg_handles, labels=leg_labels,
              fontsize=12, framealpha=0.9, edgecolor=CPX_LGREY,
              loc="upper right")

    fig.text(0.99, 0.01,
             "Cineplex Marketing Analytics  |  EXH MMM Phase 2 Iter 2",
             ha="right", va="bottom", fontsize=6, color=CPX_LGREY)
    
    # ── Attendance structure annotation ──────────────────────────────────────
    s = year_att_structure[yr]
    ann_text = (
        f"Total Attendance:                    {s['actual']/1e6:.2f}M\n"
        f"Baseline (non-media):                {s['controls']/1e6:.2f}M  "
        f"({s['controls']/s['actual']*100:.1f}% of total)\n"
        f"Media (model-validated total):       {s['media']/1e6:.2f}M  "
        f"({s['media']/s['actual']*100:.1f}% of total)\n"
        f"  of which — validated LOBs:         {s['validated']/1e6:.2f}M  "
        f"({s['validated']/s['media']*100:.1f}% of media)\n"
        f"  of which — directional LOBs ⚠:    {s['directional_graft']/1e6:.2f}M  "
        f"({s['directional_graft']/s['media']*100:.1f}% of media)"
    )
    ax.text(
        0.01, 0.97, ann_text,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=11, color=CPX_DGREY, fontweight="bold",
        fontfamily="monospace",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor=CPX_LGREY, alpha=0.95),
    )

    plt.tight_layout()
    save_fig(fig, f"I2_B9_Annual_OrigLOB_{yr}_Decomp.png")
    plt.show()
    plt.close(fig)

In [0]:
# ── Per-LOB flighting zone breakdown ─────────────────────────────────────────
df_fl = df_full[df_full["year"] < 2026].copy()  # local alias; df_fl only exists in diagnostics nb
lob_rows = []

for (funnel, lob_raw), cols in funnel_lob_cols.items():
    if not cols:
        continue

    lob_label = lob_raw.replace("_", " ")
    opt_wks   = funnel_opt_wks[funnel]
    lag_min, lag_max = funnel_lags[funnel]

    woy_lob = (
        df_fl.assign(woy=df_fl["week_date"].dt.isocalendar().week.astype(int))
             .groupby("woy")[cols]
             .mean()
             .sum(axis=1)
             .reset_index()
             .rename(columns={0: "spend"})
    )

    total = woy_lob["spend"].sum()
    if total == 0:
        continue

    sp_opt, sp_peak, sp_trough, sp_other = 0.0, 0.0, 0.0, 0.0
    for woy_row in woy_lob.itertuples():
        w, s = woy_row.woy, woy_row.spend
        if w in set(opt_wks):
            sp_opt    += s
        elif w in set(PEAK_WKS):
            sp_peak   += s
        elif w in set(TROUGH_WKS):
            sp_trough += s
        else:
            sp_other  += s

    lob_rows.append({
        "Funnel":               funnel,
        "LOB":                  lob_label,
        "Optimal %":            round(sp_opt    / total * 100, 1),
        "Neutral %":            round(sp_other  / total * 100, 1),
        "Peak wk % (too late)": round(sp_peak   / total * 100, 1),
        "Trough %":             round(sp_trough / total * 100, 1),
        "Avg wkly spend $K":    round(woy_lob["spend"].mean() / 1e3, 2),
        "Optimal lag (wks)":    f"{lag_min}–{lag_max}",
    })

lob_df = (
    pd.DataFrame(lob_rows)
    .sort_values("Optimal %", ascending=False)
    .reset_index(drop=True)
)
print("LOB Spend Zone Breakdown (funnel-specific optimal windows):")
display(lob_df)

# ── Chart ─────────────────────────────────────────────────────────────────────
MIN_AVG_SPEND_K = 0.5
lob_df["Flag"] = lob_df["Avg wkly spend $K"].lt(MIN_AVG_SPEND_K)
plot_df = lob_df[~lob_df["Flag"]].iloc[::-1].reset_index(drop=True)
n = len(plot_df)
y = np.arange(n)

fig, ax = plt.subplots(figsize=(16, max(9, n * 0.55 + 2)))
lefts = np.zeros(n)

for label, col_key, color in [
    ("Optimal",            "Optimal %",            CPX_GREEN),
    ("Neutral",            "Neutral %",            CPX_MID),
    ("Peak wk (too late)", "Peak wk % (too late)", CPX_GOLD),
    ("Trough",             "Trough %",             CPX_RED),
]:
    vals = plot_df[col_key].values
    ax.barh(y, vals, left=lefts, height=0.6, color=color, alpha=0.88, label=label)
    for i, (v, l) in enumerate(zip(vals, lefts)):
        if v >= 6:
            txt_color = "white" if color not in (CPX_MID, CPX_GOLD) else CPX_DGREY
            ax.text(l + v / 2, i, f"{v:.0f}%",
                    ha="center", va="center", fontsize=8.5, fontweight="bold",
                    color=txt_color)
    lefts += vals

for i, row in plot_df.iterrows():
    ax.text(102, i, f"${row['Avg wkly spend $K']:.1f}K/wk",
            va="center", fontsize=8, color=CPX_DGREY)

ax.axvline(100, color="#DDDDDD", lw=1)
ax.set_yticks(y)
ax.set_yticklabels(
    [f"{row['Funnel']} | {row['LOB']}" for _, row in plot_df.iterrows()],
    fontsize=9
)
ax.set_xlabel("Share of Average Weekly Spend (%)", fontsize=11)
ax.set_title(
    "EXH MMM — LOB Spend Zone Breakdown\n"
    "Sorted by % landing in optimal pre-peak window  |  Right axis = avg weekly spend",
    fontsize=13, fontweight="bold", color=CPX_NAVY,
)
ax.legend(loc="lower right", fontsize=10, frameon=True, framealpha=0.9)
ax.set_xlim(0, 120)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig(fig, "EXH_flighting_lob_zone_breakdown.png")
plt.show()

In [0]:
# ── CELL: Export all chart data to Excel — one file, multiple tabs ────────────
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage
import shutil, os

NAVY  = "002B5C"; WHITE = "FFFFFF"; LGREY = "F2F2F2"; AMBER = "FFF2CC"

hdr_font  = Font(name="Arial", bold=True, color=WHITE, size=10)
sub_font  = Font(name="Arial", bold=True, color=NAVY, size=9)
cell_font = Font(name="Arial", size=9)
dir_font  = Font(name="Arial", size=9, italic=True, color="885000")
center    = Alignment(horizontal="center", vertical="center", wrap_text=True)
left      = Alignment(horizontal="left",   vertical="center")
thin      = Side(style="thin", color="D0D0D0")
border    = Border(thin, thin, thin, thin)

FUNNEL_ROW_FILLS = {
    "Awareness"    : "EEF3FA",
    "Consideration": "FDEAEA",
    "Conversion"   : "FFFBEA",
}

def hdr_fill(color=NAVY):
    return PatternFill("solid", fgColor=color)

def write_title(ws, title, subtitle, ncols):
    ws["A1"] = title
    ws["A1"].font = Font(name="Arial", bold=True, size=12, color=NAVY)
    ws.merge_cells(f"A1:{get_column_letter(ncols)}1")
    if subtitle:
        ws["A2"] = subtitle
        ws["A2"].font = Font(name="Arial", italic=True, size=9, color="555555")
        ws.merge_cells(f"A2:{get_column_letter(ncols)}2")
    ws.append([])  # blank row
    return 4

def style_hdr_row(ws, row, cols, fill_color=NAVY):
    fill = PatternFill("solid", fgColor=fill_color)
    for j, h in enumerate(cols, 1):
        c = ws.cell(row=row, column=j, value=h)
        c.font = hdr_font; c.fill = fill
        c.alignment = center; c.border = border

def style_data_row(ws, row, values, fill_color=LGREY, is_dir=False):
    fill = PatternFill("solid", fgColor=fill_color)
    for j, v in enumerate(values, 1):
        c = ws.cell(row=row, column=j, value=v)
        c.font = dir_font if is_dir else cell_font
        c.fill = fill; c.border = border
        c.alignment = left if j <= 2 else center

def set_widths(ws, widths):
    for j, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(j)].width = w

def embed_image(ws, img_path, anchor):
    if os.path.exists(img_path):
        img = XLImage(img_path)
        img.width  = 900
        img.height = int(900 * 9 / 16)
        ws.add_image(img, anchor)
    else:
        ws[anchor] = f"Image not found: {os.path.basename(img_path)}"

wb = Workbook()
wb.remove(wb.active)

# ════════════════════════════════════════════════════════════════════════════
# TAB 1 — Decomp Table (master LOB decomposition)
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("LOB Decomp Table")
r  = write_title(ws,
    "EXH MMM — LOB Decomposition Table (Graft)",
    "Validated LOBs: contribution from model saturation curves × tilt weights. "
    "Extrapolated LOBs: funnel-avg att-per-$1 applied — directional only.",
    12)

cols = ["Funnel","LOB","Parent Combo","Status","Reallocatable",
        "Annual Spend ($)","Spend %","Tilt r","Final Share (grp)",
        "Annual Contrib","Contrib %","Att per $1"]
style_hdr_row(ws, r, cols); r += 1

for _, row in decomp.sort_values("Annual Contrib", ascending=False, na_position="last").iterrows():
    is_dir     = str(row.get("Status","")).startswith("Extrapolated")
    fill_color = AMBER if is_dir else FUNNEL_ROW_FILLS.get(row.get("Funnel",""), LGREY)
    style_data_row(ws, r, [
        row.get("Funnel",""),
        row.get("LOB",""),
        row.get("Parent Combo",""),
        row.get("Status",""),
        row.get("Reallocatable",""),
        row.get("Annual Spend", None),
        row.get("Spend %", None),
        row.get("Tilt r", None),
        row.get("Final Share (grp)", None),
        row.get("Annual Contrib", None),
        row.get("Contrib %", None),
        row.get("Att per $1", None),
    ], fill_color=fill_color, is_dir=is_dir)
    r += 1

set_widths(ws, [14,28,28,32,14,16,10,10,16,16,10,12])
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 2 — Tilt Audit
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Tilt Audit")
r  = write_title(ws,
    "Tilt Movement Audit — Pure Spend Share vs Tilted Share",
    "Shows how residual-attendance correlation adjusted each LOB's share within its group.",
    moves_df.shape[1])

style_hdr_row(ws, r, list(moves_df.columns)); r += 1
for _, row in moves_df.iterrows():
    style_data_row(ws, r, list(row.values),
                   fill_color=FUNNEL_ROW_FILLS.get(row.get("Funnel",""), LGREY))
    r += 1

for j in range(1, moves_df.shape[1] + 1):
    ws.column_dimensions[get_column_letter(j)].width = 18
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 3 — Per-LOB Spend & Contribution data
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Per-LOB Spend & Contrib")
r  = write_title(ws,
    "Per-LOB Annual Spend & Attendance Contribution by Year",
    "Contribution = Spend × att-per-$1. "
    "Directional LOBs use funnel-avg rate — flagged ⚠.",
    2 + len(years_avail) * 2)

# Header row — fixed cols + year pairs
fixed = ["Funnel", "LOB", "Directional?"]
year_subcols = []
for yr in years_avail:
    year_subcols += [f"{yr} Spend ($)", f"{yr} Contrib (att)"]

style_hdr_row(ws, r, fixed + year_subcols); r += 1

for lob in all_lobs:
    active_funnels = [f for f in FUNNELS if (f, lob) in fl_rate and
                      any(fl_yr_spend.get((f, lob), {}).get(yr, 0) > 0
                          for yr in years_avail)]
    for f in active_funnels:
        is_dir     = (f, lob) in directional_lobs
        fill_color = AMBER if is_dir else FUNNEL_ROW_FILLS.get(f, LGREY)
        row_vals   = [f, lob, "⚠ Directional" if is_dir else "Validated"]
        for yr in years_avail:
            sp = fl_yr_spend.get((f, lob), {}).get(yr, 0.0)
            ct = sp * fl_rate.get((f, lob), 0.0)
            row_vals += [round(sp, 0), round(ct, 0)]
        style_data_row(ws, r, row_vals, fill_color=fill_color, is_dir=is_dir)
        r += 1

set_widths(ws, [14, 28, 14] + [16, 18] * len(years_avail))
ws.freeze_panes = "C5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 4 — Annual LOB Contribution by Year (grouped chart data)
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Annual LOB Contrib by Year")
r  = write_title(ws,
    "Annual Media Contribution by Original LOB — All Years",
    "One row per (Funnel, LOB). Columns = contribution per year. "
    "Directional LOBs flagged — funnel-avg rate applied.",
    3 + len(years_avail))

style_hdr_row(ws, r, ["Funnel", "LOB", "Directional?"] +
              [str(yr) + " Contrib (att)" for yr in years_avail]); r += 1

for lob_item in ORIG_LOBS:
    f, lob = lob_item["funnel"], lob_item["lob"]
    is_dir = lob_item["directional"]
    fill_color = AMBER if is_dir else FUNNEL_ROW_FILLS.get(f, LGREY)
    row_vals = [f, lob, "⚠ Directional" if is_dir else "Validated"]
    for yr in years_avail:
        sp  = fl_yr_spend.get((f, lob), {}).get(yr, 0.0)
        val = sp * fl_rate.get((f, lob), 0.0)
        row_vals.append(round(val, 0))
    style_data_row(ws, r, row_vals, fill_color=fill_color, is_dir=is_dir)
    r += 1

set_widths(ws, [14, 28, 14] + [18] * len(years_avail))
ws.freeze_panes = "C5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 5 — Attendance Structure by Year
# ════════════════════════════════════════════════════════════════════════════
ws = wb.create_sheet("Attendance Structure")
r  = write_title(ws,
    "Annual Attendance Structure by Year (reconciled to actual)",
    "Media share = model total_media × recon factor. "
    "Validated/directional split uses graft extrapolated % of 2025 contrib.",
    7)

style_hdr_row(ws, r, [
    "Year", "Total Attendance", "Baseline / Non-Media",
    "Baseline %", "Total Media", "Media %",
    "  — Validated LOBs", "Validated % of Media",
    "  — Directional LOBs ⚠", "Directional % of Media"
]); r += 1

for yr in years_avail:
    s = year_att_structure[yr]
    style_data_row(ws, r, [
        yr,
        round(s["actual"], 0),
        round(s["controls"], 0),
        round(s["controls"] / s["actual"] * 100, 1),
        round(s["media"], 0),
        round(s["media"] / s["actual"] * 100, 1),
        round(s["validated"], 0),
        round(s["validated"] / s["media"] * 100, 1),
        round(s["directional_graft"], 0),
        round(s["directional_graft"] / s["media"] * 100, 1),
    ], fill_color="EEF3FA")
    r += 1

set_widths(ws, [8, 18, 20, 14, 16, 10, 18, 20, 20, 22])
ws.freeze_panes = "A5"

# ════════════════════════════════════════════════════════════════════════════
# TAB 6 — Per-LOB charts (one image tab per LOB)
# ════════════════════════════════════════════════════════════════════════════
for lob in all_lobs:
    safe_name = lob.replace(" ", "_").replace("/", "_").replace("&", "and")
    img_path  = os.path.join(OUT_PATH, f"I2_B9_LOB_{safe_name}_SpendContrib.png")
    tab_name  = lob[:28].strip()  # Excel tab name limit = 31 chars
    ws = wb.create_sheet(tab_name)
    ws["A1"] = f"{lob} — Annual Spend & Attendance Contribution"
    ws["A1"].font = Font(name="Arial", bold=True, size=12, color=NAVY)
    embed_image(ws, img_path, "A3")

# ════════════════════════════════════════════════════════════════════════════
# TAB 7 — Annual grouped charts (one per year)
# ════════════════════════════════════════════════════════════════════════════
for yr in years_avail:
    img_path = os.path.join(OUT_PATH, f"I2_B9_Annual_OrigLOB_{yr}_Decomp.png")
    ws = wb.create_sheet(f"Annual LOBs {yr}")
    ws["A1"] = f"Annual Media Contribution by LOB — {yr}"
    ws["A1"].font = Font(name="Arial", bold=True, size=12, color=NAVY)
    embed_image(ws, img_path, "A3")

# ════════════════════════════════════════════════════════════════════════════
# Save
# ════════════════════════════════════════════════════════════════════════════
_tmp      = "/tmp/EXH_MMM_LOB_Decomp_Charts_Data.xlsx"
xlsx_path = os.path.join(OUT_PATH, "EXH_MMM_LOB_Decomp_Charts_Data.xlsx")
wb.save(_tmp)
shutil.copy2(_tmp, xlsx_path)
os.remove(_tmp)
print(f"✓ Saved: {xlsx_path}")
print(f"  Tabs: LOB Decomp Table | Tilt Audit | Per-LOB Spend & Contrib | "
      f"Annual LOB Contrib by Year | Attendance Structure | "
      f"{len(all_lobs)} LOB chart tabs | {len(years_avail)} Annual chart tabs")